<div style="padding: 20px; background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">📈 Module 3.1: The Mathematics of Embeddings</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">A comprehensive deep-dive into how machine learning models understand human text.</p>
</div>

---

## 1. What Exactly is an Embedding?

At its core, an **embedding** is a translation. It translates human concepts (words, sentences, images) into lists of numbers (vectors) that a computer can perform math on.

### Dense vs. Sparse Vectors
- **Sparse Vectors (Traditional ML):** Methods like TF-IDF or One-Hot Encoding create vectors where almost all values are `0`. If your vocabulary has 50,000 words, each word is a 50,000-dimensional vector with a single `1`. This is extremely inefficient and captures **zero semantic meaning**.
- **Dense Vectors (Deep Learning):** Embeddings are *dense*. They usually have a fixed size (e.g., 384, 768, or 1536 dimensions) where *every* number is a floating-point value. Instead of representing the *presence* of a word, they represent its **meaning in a latent semantic space**.

> [!TIP]
> **Proximity = Similarity**. The golden rule of embeddings is that if two concepts are semantically related (like "Dog" and "Puppy"), their corresponding vectors will be geometrically close to each other in the high-dimensional space.

### Course alignment and free-first stack

- Covers: Embedding vectors, semantic similarity, distance metrics, and visualization.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
# Let's load a free, local embedding model from HuggingFace
# We use sentence-transformers which is the industry standard for this.
from sentence_transformers import SentenceTransformer
import numpy as np
import os

# Disable tokenizer parallelism warning
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Load a lightweight, fast model that works purely on CPU
print("Loading model (this might take a few seconds on first run)...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Let's see the vector size (dimensions) of this model
sample_embedding = model.encode("Hello World")
print(f"Model loaded successfully.")
print(f"This model produces vectors with {len(sample_embedding)} dimensions.")
print(f"First 5 values of the vector: {sample_embedding[:5]}")

## 2. Distance Metrics: The Math of Similarity

Once we have vectors, how do we know if they are "close"? We use distance metrics. In NLP, **Cosine Similarity** is the undisputed king.

### A. Cosine Similarity
$\text{Cosine Similarity} = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$

Cosine similarity measures the **angle** between two vectors, ignoring their magnitude (length). It ranges from `-1` (perfect opposites) to `1` (identical direction). For text, we almost always want to know if texts point in the same semantic direction, regardless of how long the texts are.

### B. Dot Product
$\text{Dot Product} = \mathbf{A} \cdot \mathbf{B} = \sum_{i=1}^{n} a_i b_i$

Dot product is extremely fast to compute. *If your vectors are normalized* (meaning their length is scaled to 1), then the Dot Product is mathematically identical to Cosine Similarity. Most modern models output normalized vectors precisely so we can use the ultra-fast Dot Product for search.

### C. Euclidean Distance (L2)
$\text{Euclidean} = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$

This is straight-line distance. It is sensitive to magnitude. It's rarely used for text embeddings because a long document and a short summary might have similar meanings (same angle) but very different magnitudes.

In [ ]:
# Let's implement these metrics in pure NumPy to see how they work

def calculate_cosine_similarity(vec_a, vec_b):
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot_product / (norm_a * norm_b)

# 1. Define three sentences
sentence_1 = "The golden retriever is playing fetch in the park."
sentence_2 = "A happy dog is running after a ball outside." # Highly related to 1
sentence_3 = "The stock market saw a massive crash today due to inflation fears." # Unrelated

# 2. Convert to embeddings
emb_1 = model.encode(sentence_1)
emb_2 = model.encode(sentence_2)
emb_3 = model.encode(sentence_3)

# 3. Calculate similarities
sim_1_2 = calculate_cosine_similarity(emb_1, emb_2)
sim_1_3 = calculate_cosine_similarity(emb_1, emb_3)

print(f"Cosine Similarity (Dog vs Dog): {sim_1_2:.4f}")
print(f"Cosine Similarity (Dog vs Stocks): {sim_1_3:.4f}")

# Notice how 1 and 2 score much higher than 1 and 3.
# This is the exact mechanism that powers Vector Search in RAG pipelines!

## 3. Visualizing High-Dimensional Space (PCA)

Our model uses 384 dimensions. Humans can only see in 3D. How do we visualize this?
We use **Principal Component Analysis (PCA)**, an algorithm that "squashes" high-dimensional data down to 2D while trying to preserve the distance relationships between points.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Let's create a distinct vocabulary with clear categories
vocabulary = [
    # Animals
    "Dog", "Puppy", "Cat", "Kitten", "Wolf", "Tiger",
    # Finance
    "Stock", "Bond", "Bank", "Money", "Investment", "Economy",
    # Technology
    "Computer", "Laptop", "Software", "Algorithm", "Internet", "Server"
]

# Encode all words
word_embeddings = model.encode(vocabulary)

# Squash 384 dimensions down to 2 dimensions using PCA
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(word_embeddings)

# Plotting
plt.figure(figsize=(12, 8))
plt.style.use('seaborn-v0_8-whitegrid')

# Scatter plot
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], c='#e74c3c', s=150, alpha=0.8, edgecolors='black')

# Add labels to the points
for i, word in enumerate(vocabulary):
    plt.annotate(
        word, 
        (vectors_2d[i, 0], vectors_2d[i, 1]),
        xytext=(8, 8), 
        textcoords='offset points',
        fontsize=12,
        fontweight='bold'
    )

plt.title("2D PCA Visualization of Semantic Relationships", fontsize=16, fontweight='bold')
plt.xlabel("Principal Component 1 (Greatest Variance)", fontsize=12)
plt.ylabel("Principal Component 2", fontsize=12)

# You should clearly see clusters forming automatically based purely on meaning!
plt.show()